In [0]:
%pip install requests beautifulsoup4
dbutils.library.restartPython()


In [0]:
from pyspark.sql.functions import current_timestamp


In [0]:
import re, csv, json, time
from datetime import datetime
from urllib.parse import urljoin, urlparse
from zoneinfo import ZoneInfo
import io
import requests
from bs4 import BeautifulSoup

# -------------------- Config --------------------
BASE_URL = "https://www.sanjeevkapoor.com"
LIST_URL_TEMPLATE = BASE_URL + "/veg?page={}"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; SanjeevKapoorDatabricksPipeline/1.0)"}

CATALOG = "food_ops"
SCHEMA = "ingestion"
VOLUME = "sanjeev_files"

BASE_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/sanjeevkapoor_pipeline"
DATA_DIR = f"{BASE_DIR}/data"
DAILY_DIR = f"{DATA_DIR}/daily"
STATE_FILE = f"{DATA_DIR}/state.json"
CSV_FILE = f"{DATA_DIR}/recipes.csv"

TABLE_NAME = f"{CATALOG}.{SCHEMA}.sanjeev_recipes"

COLUMNS = [
    "recipe_name",
    "ingredients",
    "methods",
    "cuisines",
    "prep_time",
    "cook_time",
    "others",
]

# Optional parameters via widgets (for Jobs "Parameters")
try:
    dbutils.widgets.text("start_page", "1")
    dbutils.widgets.text("max_pages", "5")
    dbutils.widgets.text("delay_sec", "0.8")
    start_page = int(dbutils.widgets.get("start_page"))
    max_pages = int(dbutils.widgets.get("max_pages"))
    delay_sec = float(dbutils.widgets.get("delay_sec"))
except Exception:
    start_page, max_pages, delay_sec = 1, 5, 0.8


# -------------------- Helpers --------------------
def ensure_dirs():
    dbutils.fs.mkdirs(DATA_DIR)
    dbutils.fs.mkdirs(DAILY_DIR)

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def read_text(path: str):
    try:
        return dbutils.fs.head(path, 10_000_000)
    except Exception:
        return None

def write_text(path: str, text: str):
    dbutils.fs.put(path, text, overwrite=True)

def load_state():
    ensure_dirs()
    raw = read_text(STATE_FILE)
    if not raw:
        return {"seen_urls": [], "records": [], "last_run_at": None}
    data = json.loads(raw)
    data.setdefault("seen_urls", [])
    data.setdefault("records", [])
    return data

def save_state(state):
    state["last_run_at"] = datetime.now(tz=ZoneInfo("Asia/Kolkata")).isoformat()
    write_text(STATE_FILE, json.dumps(state, ensure_ascii=False, indent=2))

# def write_csv(path: str, rows: list):
#     # write to local temp then copy to volume
#     local_tmp = "/tmp/sanjeev_recipes_tmp.csv"
#     with open(local_tmp, "w", encoding="utf-8-sig", newline="") as f:
#         w = csv.DictWriter(f, fieldnames=COLUMNS)
#         w.writeheader()
#         for r in rows:
#             w.writerow({k: r.get(k, "") for k in COLUMNS})
#     dbutils.fs.cp("file:" + local_tmp, path, True)


def write_csv(path: str, rows: list):
    buffer = io.StringIO()
    w = csv.DictWriter(buffer, fieldnames=COLUMNS)
    w.writeheader()
    for r in rows:
        w.writerow({k: r.get(k, "") for k in COLUMNS})
    dbutils.fs.put(path, buffer.getvalue(), True)

def get_html(url, timeout=30):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r.text

def normalize_recipe_url(href: str):
    if not href:
        return None
    url = urljoin(BASE_URL, href.split("#")[0].split("?")[0])
    u = urlparse(url)
    if u.netloc != urlparse(BASE_URL).netloc:
        return None
    if "/Recipe/" not in u.path:
        return None
    return f"{u.scheme}://{u.netloc}{u.path}"

def extract_recipe_links(list_html: str):
    soup = BeautifulSoup(list_html, "html.parser")
    links = set()
    for a in soup.find_all("a", href=True):
        u = normalize_recipe_url(a["href"])
        if u:
            links.add(u)
    return sorted(links)

def extract_table_map(soup: BeautifulSoup):
    kv = {}
    post = soup.select_one("#post-container") or soup
    table = post.find("table")
    if not table:
        return kv
    for tr in table.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) >= 2:
            k = clean_text(tds[0].get_text(" ", strip=True)).lower()
            v = clean_text(tds[1].get_text(" ", strip=True))
            kv[k] = v
    return kv

def list_after_h2(soup: BeautifulSoup, heading_text: str):
    for h2 in soup.find_all("h2"):
        if clean_text(h2.get_text(" ", strip=True)).lower() == heading_text.lower():
            nxt = h2.find_next_sibling()
            while nxt and getattr(nxt, "name", None) not in ("ul", "ol"):
                nxt = nxt.find_next_sibling()
            if nxt and nxt.name in ("ul", "ol"):
                return [clean_text(li.get_text(" ", strip=True)) for li in nxt.find_all("li")]
    return []

def parse_recipe(recipe_url: str, html: str):
    soup = BeautifulSoup(html, "html.parser")

    name = ""
    h1 = soup.find("h1")
    if h1:
        name = clean_text(h1.get_text(" ", strip=True))
    if not name:
        og = soup.find("meta", attrs={"property": "og:title"})
        if og and og.get("content"):
            name = clean_text(og["content"])

    table = extract_table_map(soup)
    ingredients = list_after_h2(soup, "Ingredients")
    methods = list_after_h2(soup, "Method")

    others = table.get("others", "")
    if not others:
        extra = []
        for k, v in table.items():
            if k not in {"main ingredients", "cuisine", "prep time", "cook time", "others"}:
                extra.append(f"{k}: {v}")
        others = "; ".join(extra)

    return {
        "recipe_name": name,
        "ingredients": " | ".join(ingredients),
        "methods": " | ".join(methods),
        "cuisines": table.get("cuisine", ""),
        "prep_time": table.get("prep time", ""),
        "cook_time": table.get("cook time", ""),
        "others": others,
        "source_url": recipe_url,
    }


# -------------------- Main Pipeline --------------------
def run_pipeline(start_page=1, max_pages=5, delay_sec=0.8):
    state = load_state()
    seen = set(state["seen_urls"])
    all_links = set()

    print(f"Starting: start_page={start_page}, max_pages={max_pages}")

    for p in range(start_page, start_page + max_pages):
        url = LIST_URL_TEMPLATE.format(p)
        try:
            html = get_html(url)
            links = extract_recipe_links(html)
            print(f"Page {p}: {len(links)} links")
            all_links.update(links)
            if p > start_page and len(links) == 0:
                break
            if delay_sec:
                time.sleep(delay_sec)
        except Exception as e:
            print(f"Page {p} failed: {e}")

    new_links = [u for u in sorted(all_links) if u not in seen]
    print("Total discovered:", len(all_links))
    print("New links:", len(new_links))

    new_records = []
    for i, u in enumerate(new_links, 1):
        try:
            html = get_html(u)
            rec = parse_recipe(u, html)
            if rec["recipe_name"]:
                state["records"].append({k: rec.get(k, "") for k in COLUMNS})
                new_records.append(rec)
                seen.add(u)
                print(f"[{i}/{len(new_links)}] Added: {rec['recipe_name']}")
            else:
                print(f"[{i}/{len(new_links)}] Skipped: {u}")
            if delay_sec:
                time.sleep(delay_sec)
        except Exception as e:
            print(f"[{i}/{len(new_links)}] Failed: {u} -> {e}")

    state["seen_urls"] = sorted(seen)
    save_state(state)

    # Full CSV
    write_csv(CSV_FILE, state["records"])

    # Daily new CSV (IST date)
    daily_file = None
    if new_records:
        ist_date = datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d")
        daily_file = f"{DAILY_DIR}/new_{ist_date}.csv"
        write_csv(daily_file, [{k: r.get(k, "") for k in COLUMNS} for r in new_records])

        # Optional append to Delta table
        spark.createDataFrame(new_records).withColumn("ingested_at", current_timestamp()) \
            .select("recipe_name","ingredients","methods","cuisines","prep_time","cook_time","others","source_url","ingested_at") \
            .dropDuplicates(["source_url"]) \
            .write.mode("append").saveAsTable(TABLE_NAME)

    print("Run complete")
    print("New rows added:", len(new_records))
    print("CSV:", CSV_FILE)
    print("Daily CSV:", daily_file if daily_file else "No new rows")

run_pipeline(start_page=start_page, max_pages=max_pages, delay_sec=delay_sec)
